In [0]:
catalog = "databrickscaresync"

table = f"{catalog}.silver.lab_results"

print("========== LAB RESULTS SCHEMA ==========")
print(spark.table(table).columns)

print("\n========== LAB RESULTS SAMPLE ==========")
spark.table(table).show(5, truncate=False)

print("\n========== REQUIRED COLUMNS ==========")
required = [
    "hospital_id",
    "patient_id",
    "doctor_id",
    "test_date",
    "result_status"
]

actual = spark.table(table).columns

for column in required:
    print(f"{column}: {'FOUND' if column in actual else 'MISSING'}")

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DateType, BooleanType, DecimalType
)

# ============================================================
# STEP 1: CONFIGURATION
# ============================================================

catalog = "databrickscaresync"

source_table = f"{catalog}.silver.lab_results"
hospital_table = f"{catalog}.silver.hospitals"
calendar_table = f"{catalog}.silver.calendar"

target_table = f"{catalog}.gold.hospital_daily_summary"

print("[SOURCE]", source_table)
print("[TARGET]", target_table)


# ============================================================
# STEP 2: VALIDATE SOURCE TABLES
# ============================================================

if not spark.catalog.tableExists(source_table):
    raise ValueError(f"Source table does not exist: {source_table}")

if not spark.catalog.tableExists(hospital_table):
    raise ValueError(f"Hospital table does not exist: {hospital_table}")

if not spark.catalog.tableExists(calendar_table):
    raise ValueError(f"Calendar table does not exist: {calendar_table}")

lab_df = spark.table(source_table)
hospital_df = spark.table(hospital_table)
calendar_df = spark.table(calendar_table)

required_lab_columns = [
    "hospital_id",
    "patient_id",
    "doctor_id",
    "test_date",
    "result_status"
]

missing_columns = [
    c for c in required_lab_columns
    if c not in lab_df.columns
]

if missing_columns:
    raise ValueError(
        f"Cannot build hospital_daily_summary. "
        f"{source_table} is missing required columns: {missing_columns}. "
        f"No target table was modified."
    )

required_hospital_columns = ["hospital_id", "hospital_name"]

missing_hospital = [
    c for c in required_hospital_columns
    if c not in hospital_df.columns
]

if missing_hospital:
    raise ValueError(
        f"Hospital table is missing columns: {missing_hospital}. "
        f"No target table was modified."
    )

required_calendar_columns = [
    "date_key", "date", "day_name", "is_weekend"
]

missing_calendar = [
    c for c in required_calendar_columns
    if c not in calendar_df.columns
]

if missing_calendar:
    raise ValueError(
        f"Calendar table is missing columns: {missing_calendar}. "
        f"No target table was modified."
    )

print("[VALIDATION PASSED] Source schemas are valid.")


# ============================================================
# STEP 3: VALIDATE SOURCE DATA
# ============================================================

if lab_df.limit(1).count() == 0:
    raise ValueError(
        "Lab results table is empty. No target table was modified."
    )

if hospital_df.groupBy("hospital_id").count().filter(
    F.col("count") > 1
).limit(1).count() > 0:
    raise ValueError(
        "Duplicate hospital_id values found. "
        "Hospital join could duplicate summary rows."
    )

if calendar_df.groupBy("date_key").count().filter(
    F.col("count") > 1
).limit(1).count() > 0:
    raise ValueError(
        "Duplicate calendar date_key values found."
    )

print("[DATA VALIDATION PASSED]")


# ============================================================
# STEP 4: AGGREGATE LAB RESULTS
# Grain: One row per hospital per day
# ============================================================

daily_df = (
    lab_df
    .withColumn("test_date", F.to_date("test_date"))
    .groupBy(
        "hospital_id",
        "test_date"
    )
    .agg(
        F.count("*").alias("total_tests"),

        F.countDistinct("patient_id")
        .alias("unique_patients_tested"),

        F.countDistinct("doctor_id")
        .alias("active_doctor_count"),

        F.sum(
            F.when(
                F.col("result_status") == "Abnormal", 1
            ).otherwise(0)
        ).alias("abnormal_test_count"),

        F.sum(
            F.when(
                F.col("result_status") == "Critical", 1
            ).otherwise(0)
        ).alias("critical_test_count")
    )
)


# ============================================================
# STEP 5: CALCULATE ABNORMAL AND CRITICAL RATES
# ============================================================

daily_df = (
    daily_df
    .withColumn(
        "abnormal_rate_pct",
        F.round(
            F.col("abnormal_test_count") /
            F.col("total_tests") * 100,
            2
        ).cast(DecimalType(5, 2))
    )
    .withColumn(
        "critical_rate_pct",
        F.round(
            F.col("critical_test_count") /
            F.col("total_tests") * 100,
            2
        ).cast(DecimalType(5, 2))
    )
)


# ============================================================
# STEP 6: JOIN HOSPITAL INFORMATION
# ============================================================

daily_df = (
    daily_df.alias("d")
    .join(
        hospital_df.select(
            "hospital_id",
            "hospital_name"
        ).alias("h"),
        F.col("d.hospital_id") == F.col("h.hospital_id"),
        "left"
    )
    .select(
        F.col("d.hospital_id").cast("int").alias("hospital_id"),
        F.col("h.hospital_name").alias("hospital_name"),
        F.col("d.test_date").alias("summary_date"),
        F.col("d.total_tests"),
        F.col("d.unique_patients_tested"),
        F.col("d.active_doctor_count"),
        F.col("d.abnormal_test_count"),
        F.col("d.critical_test_count"),
        F.col("d.abnormal_rate_pct"),
        F.col("d.critical_rate_pct")
    )
)


# ============================================================
# STEP 7: JOIN CALENDAR INFORMATION
# ============================================================

calendar_lookup = (
    calendar_df
    .select(
        F.to_date("date").alias("calendar_date"),
        "day_name",
        "is_weekend"
    )
)

daily_df = (
    daily_df.alias("d")
    .join(
        calendar_lookup.alias("c"),
        F.col("d.summary_date") == F.col("c.calendar_date"),
        "left"
    )
    .select(
        "d.*",
        F.col("c.day_name").alias("day_name"),
        F.col("c.is_weekend").alias("is_weekend")
    )
)


# ============================================================
# STEP 8: CALCULATE PREVIOUS DAY TEST VOLUME
# ============================================================

window_spec = (
    Window
    .partitionBy("hospital_id")
    .orderBy("summary_date")
)

daily_df = (
    daily_df
    .withColumn(
        "prev_day_total_tests",
        F.lag("total_tests").over(window_spec)
    )
    .withColumn(
        "test_volume_change_pct",
        F.when(
            F.col("prev_day_total_tests").isNull() |
            (F.col("prev_day_total_tests") == 0),
            F.lit(None)
        ).otherwise(
            F.round(
                (
                    F.col("total_tests") -
                    F.col("prev_day_total_tests")
                ) / F.col("prev_day_total_tests") * 100,
                2
            )
        ).cast(DecimalType(6, 2))
    )
)


# ============================================================
# STEP 9: SELECT FINAL STTM COLUMNS
# ============================================================

final_df = daily_df.select(
    F.col("hospital_id").cast(IntegerType()),
    F.col("hospital_name").cast(StringType()),
    F.col("summary_date").cast(DateType()),
    F.col("day_name").cast(StringType()),
    F.col("is_weekend").cast(BooleanType()),
    F.col("total_tests").cast(IntegerType()),
    F.col("unique_patients_tested").cast(IntegerType()),
    F.col("active_doctor_count").cast(IntegerType()),
    F.col("abnormal_test_count").cast(IntegerType()),
    F.col("critical_test_count").cast(IntegerType()),
    F.col("abnormal_rate_pct").cast(DecimalType(5, 2)),
    F.col("critical_rate_pct").cast(DecimalType(5, 2)),
    F.col("prev_day_total_tests").cast(IntegerType()),
    F.col("test_volume_change_pct").cast(DecimalType(6, 2))
)


# ============================================================
# STEP 10: VALIDATE FINAL DATA BEFORE WRITING
# ============================================================

if final_df.limit(1).count() == 0:
    raise ValueError(
        "Final daily summary is empty. No target table was modified."
    )

if final_df.groupBy(
    "hospital_id", "summary_date"
).count().filter(
    F.col("count") > 1
).limit(1).count() > 0:
    raise ValueError(
        "Duplicate hospital_id + summary_date grain detected. "
        "No target table was modified."
    )

print("[FINAL VALIDATION PASSED]")
print("Rows to write:", final_df.count())


# ============================================================
# STEP 11: WRITE TARGET TABLE
# ============================================================
# Full rebuild as specified by STTM.
# This is the only section that modifies an existing Gold table.

if spark.catalog.tableExists(target_table):
    raise RuntimeError(
        f"Target already exists: {target_table}. "
        "To protect existing data, this notebook will not overwrite it. "
        "Review and explicitly approve the replacement before proceeding."
    )

(
    final_df.write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable(target_table)
)

print(f"[DONE] Created {target_table}")


# ============================================================
# STEP 12: POST-WRITE VALIDATION
# ============================================================

result_df = spark.table(target_table)

print("\n========== FINAL TABLE VALIDATION ==========")

result_df.printSchema()

result_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("hospital_id", "summary_date")
    .alias("distinct_hospital_days"),
    F.sum("total_tests").alias("total_tests")
).show()

print("\n========== SAMPLE RECORDS ==========")

result_df.orderBy(
    "hospital_id", "summary_date"
).show(10, truncate=False)


In [0]:
from pyspark.sql import functions as F

catalog = "databrickscaresync"

tables = [
    "patients",
    "hospitals",
    "doctors",
    "insurance_providers",
    "lab_results"
]

for name in tables:
    print(f"\n========== {name.upper()} ==========")

    for layer in ["bronze", "silver"]:
        table = f"{catalog}.{layer}.{name}"
        df = spark.table(table)

        print(f"\n{layer.upper()} | Rows: {df.count()}")

        if "landing_timestamp" in df.columns:
            df.groupBy("landing_timestamp") \
                .count() \
                .orderBy(F.col("landing_timestamp").desc()) \
                .show(10, truncate=False)

In [0]:

# ============================================================
# NB_hospital_daily_summary
# Purpose: Hospital-level daily laboratory test summary
# Target: databrickscaresync.gold.hospital_daily_summary
# Grain: One row per hospital per day
# Load strategy: FULL REBUILD
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# STEP 1: CONFIGURATION
# ============================================================

catalog = "databrickscaresync"

source_table = f"{catalog}.silver.lab_results"
hospital_table = f"{catalog}.silver.hospitals"
calendar_table = f"{catalog}.silver.calendar"

target_table = f"{catalog}.gold.hospital_daily_summary"

print(f"[SOURCE] {catalog}.silver")
print(f"[TARGET] {target_table}")


# ============================================================
# STEP 2: READ SOURCE TABLES
# ============================================================

lab_df = spark.table(source_table)
hospital_df = spark.table(hospital_table)
calendar_df = spark.table(calendar_table)


# ============================================================
# STEP 3: SCHEMA VALIDATION
# ============================================================

required_lab_columns = [
    "hospital_id",
    "patient_id",
    "doctor_id",
    "test_date",
    "result_status"
]

required_hospital_columns = [
    "hospital_id",
    "hospital_name"
]

required_calendar_columns = [
    "date_key",
    "date",
    "day_name",
    "is_weekend"
]

missing_lab = [
    c for c in required_lab_columns
    if c not in lab_df.columns
]

missing_hospital = [
    c for c in required_hospital_columns
    if c not in hospital_df.columns
]

missing_calendar = [
    c for c in required_calendar_columns
    if c not in calendar_df.columns
]

if missing_lab:
    raise ValueError(
        f"Schema mismatch in {source_table}. "
        f"Missing columns: {missing_lab}"
    )

if missing_hospital:
    raise ValueError(
        f"Schema mismatch in {hospital_table}. "
        f"Missing columns: {missing_hospital}"
    )

if missing_calendar:
    raise ValueError(
        f"Schema mismatch in {calendar_table}. "
        f"Missing columns: {missing_calendar}"
    )

print("[VALIDATION PASSED]")


# ============================================================
# STEP 4: PREPARE LAB RESULTS
# ============================================================

lab_df = lab_df.withColumn(
    "test_date",
    F.to_date("test_date")
)


# ============================================================
# STEP 5: AGGREGATE DAILY HOSPITAL METRICS
# ============================================================

daily_summary_df = (
    lab_df
    .groupBy(
        "hospital_id",
        "test_date"
    )
    .agg(
        F.count("*").alias("total_tests"),

        F.countDistinct("patient_id")
        .alias("unique_patients_tested"),

        F.countDistinct("doctor_id")
        .alias("active_doctor_count"),

        F.sum(
            F.when(
                F.col("result_status") == "Abnormal", 1
            ).otherwise(0)
        ).alias("abnormal_test_count"),

        F.sum(
            F.when(
                F.col("result_status") == "Critical", 1
            ).otherwise(0)
        ).alias("critical_test_count")
    )
)


# ============================================================
# STEP 6: CALCULATE TEST RATES
# ============================================================

daily_summary_df = (
    daily_summary_df
    .withColumn(
        "abnormal_rate_pct",
        F.round(
            F.col("abnormal_test_count") /
            F.col("total_tests") * 100,
            2
        )
    )
    .withColumn(
        "critical_rate_pct",
        F.round(
            F.col("critical_test_count") /
            F.col("total_tests") * 100,
            2
        )
    )
)


# ============================================================
# STEP 7: JOIN HOSPITAL DETAILS
# ============================================================

daily_summary_df = (
    daily_summary_df.alias("l")
    .join(
        hospital_df.select(
            "hospital_id",
            "hospital_name"
        ).alias("h"),
        F.col("l.hospital_id") == F.col("h.hospital_id"),
        "left"
    )
    .select(
        F.col("l.hospital_id"),
        F.col("h.hospital_name"),
        F.col("l.test_date").alias("summary_date"),
        F.col("l.total_tests"),
        F.col("l.unique_patients_tested"),
        F.col("l.active_doctor_count"),
        F.col("l.abnormal_test_count"),
        F.col("l.critical_test_count"),
        F.col("l.abnormal_rate_pct"),
        F.col("l.critical_rate_pct")
    )
)


# ============================================================
# STEP 8: JOIN CALENDAR DETAILS
# ============================================================

calendar_lookup = calendar_df.select(
    F.to_date("date").alias("calendar_date"),
    "day_name",
    "is_weekend"
)

daily_summary_df = (
    daily_summary_df.alias("d")
    .join(
        calendar_lookup.alias("c"),
        F.col("d.summary_date") == F.col("c.calendar_date"),
        "left"
    )
    .select(
        "d.*",
        F.col("c.day_name"),
        F.col("c.is_weekend")
    )
)


# ============================================================
# STEP 9: CALCULATE PREVIOUS DAY TEST VOLUME
# ============================================================

hospital_window = (
    Window
    .partitionBy("hospital_id")
    .orderBy("summary_date")
)

daily_summary_df = (
    daily_summary_df
    .withColumn(
        "prev_day_total_tests",
        F.lag("total_tests").over(hospital_window)
    )
)


# ============================================================
# STEP 10: CALCULATE DAY-OVER-DAY CHANGE %
# ============================================================

daily_summary_df = (
    daily_summary_df
    .withColumn(
        "test_volume_change_pct",
        F.when(
            F.col("prev_day_total_tests").isNull() |
            (F.col("prev_day_total_tests") == 0),
            F.lit(None)
        ).otherwise(
            F.round(
                (
                    F.col("total_tests") -
                    F.col("prev_day_total_tests")
                ) / F.col("prev_day_total_tests") * 100,
                2
            )
        )
    )
)


# ============================================================
# STEP 11: SELECT FINAL STTM COLUMNS
# ============================================================

final_df = daily_summary_df.select(
    F.col("hospital_id").cast("int"),
    F.col("hospital_name").cast("string"),
    F.col("summary_date").cast("date"),
    F.col("day_name").cast("string"),
    F.col("is_weekend").cast("boolean"),
    F.col("total_tests").cast("int"),
    F.col("unique_patients_tested").cast("int"),
    F.col("active_doctor_count").cast("int"),
    F.col("abnormal_test_count").cast("int"),
    F.col("critical_test_count").cast("int"),
    F.col("abnormal_rate_pct").cast("decimal(5,2)"),
    F.col("critical_rate_pct").cast("decimal(5,2)"),
    F.col("prev_day_total_tests").cast("int"),
    F.col("test_volume_change_pct").cast("decimal(6,2)")
)


# ============================================================
# STEP 12: FINAL VALIDATION
# ============================================================

if final_df.limit(1).count() == 0:
    raise ValueError("No records generated for hospital_daily_summary")

duplicate_rows = (
    final_df
    .groupBy("hospital_id", "summary_date")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_rows.limit(1).count() > 0:
    raise ValueError(
        "Duplicate hospital_id + summary_date records detected"
    )

print("[FINAL VALIDATION PASSED]")
print("[ROWS TO WRITE]", final_df.count())


# ============================================================
# STEP 13: FULL REBUILD
# ============================================================
# Refuse to overwrite an existing table without approval.

if spark.catalog.tableExists(target_table):
    raise RuntimeError(
        f"Target already exists: {target_table}. "
        "Existing data is protected. Review and approve replacement first."
    )

final_df.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable(target_table)

print(f"[DONE] Created {target_table}")


# ============================================================
# STEP 14: POST-LOAD VALIDATION
# ============================================================

result_df = spark.table(target_table)

result_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct(
        "hospital_id", "summary_date"
    ).alias("distinct_hospital_days"),
    F.sum("total_tests").alias("total_tests")
).show()

result_df.orderBy(
    "hospital_id", "summary_date"
).show(10, truncate=False)

print("[SUCCESS] hospital_daily_summary completed.")


In [0]:
catalog = "databrickscaresync"

print("========== SILVER TABLES ==========")

for row in spark.sql(
    f"SHOW TABLES IN {catalog}.silver"
).collect():
    table_name = row.tableName
    full_name = f"{catalog}.silver.{table_name}"

    print(f"\n--- {table_name} ---")
    print("Columns:", spark.table(full_name).columns)
    print("Rows:", spark.table(full_name).count())

In [0]:
catalog = dbutils.widgets.get("catalog").strip()

for layer in ["bronze", "silver"]:
    schema = f"{catalog}.{layer}"

    print(f"\n{'='*65}")
    print(f"{layer.upper()} LAB RESULTS")
    print("="*65)

    table = f"{schema}.lab_results"

    print("Schema:")
    spark.sql(f"DESCRIBE TABLE {table}").show(100, False)

    print("Sample records:")
    spark.table(table).show(10, False)

    print("Row count:")
    print(spark.table(table).count())

In [0]:

catalog = dbutils.widgets.get("catalog").strip()
source_schema = f"{catalog}.silver"

tables = spark.sql(
    f"SHOW TABLES IN {source_schema}"
).collect()

for row in tables:
    table_name = row.tableName
    full_table = f"{source_schema}.{table_name}"

    print(f"\n{'='*70}")
    print(f"TABLE: {full_table}")
    print("="*70)

    df = spark.table(full_table)

    print("COLUMNS:", df.columns)

    display(df.limit(2))


In [0]:

from pyspark.sql import functions as F

catalog = "databrickscaresync"

required_columns = {
    "patient_id",
    "hospital_id",
    "doctor_id",
    "test_date",
    "result_status"
}

print("========== SEARCHING EXISTING TABLES ==========")

for schema in ["bronze", "silver"]:
    print(f"\n========== {schema.upper()} ==========")

    tables = spark.sql(
        f"SHOW TABLES IN {catalog}.{schema}"
    ).collect()

    for row in tables:
        name = row.tableName
        full_name = f"{catalog}.{schema}.{name}"

        df = spark.table(full_name)
        columns = set(df.columns)

        if required_columns.issubset(columns):
            print(f"[POTENTIAL LAB SOURCE] {full_name}")
            print("Rows:", df.count())
            print("Columns:", df.columns)

        elif "test_date" in columns or "result_status" in columns:
            print(f"[PARTIAL MATCH] {full_name}")
            print("Columns:", df.columns)

print("\n========== SEARCH COMPLETE ==========")
print("No tables were modified.")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType, DateType, BooleanType, DecimalType

CATALOG = "databrickscaresync"
LAB = f"{CATALOG}.silver.lab_results"
HOSPITALS = f"{CATALOG}.silver.hospitals"
CALENDAR = f"{CATALOG}.silver.calendar"
TARGET = f"{CATALOG}.gold.hospital_daily_summary"

ALLOW_CREATE_TARGET = False

print("Hospital Daily Summary Gold build")
print("Target:", TARGET)


In [0]:
lab_df = spark.table(LAB)
hospital_df = spark.table(HOSPITALS)
calendar_df = spark.table(CALENDAR)

required_lab = {"hospital_id", "patient_id", "doctor_id", "test_date", "result_status"}
required_hospitals = {"hospital_id", "hospital_name"}
required_calendar = {"date_key", "date", "day_name", "is_weekend"}

checks = {
    LAB: (set(lab_df.columns), required_lab),
    HOSPITALS: (set(hospital_df.columns), required_hospitals),
    CALENDAR: (set(calendar_df.columns), required_calendar),
}
errors = []
for table_name, (actual, required) in checks.items():
    missing = sorted(required - actual)
    print(table_name, "columns:", sorted(actual))
    if missing:
        errors.append(f"{table_name} missing required columns: {missing}")

if errors:
    for error in errors:
        print("SCHEMA ERROR:", error)
    raise ValueError(
        "Source schema validation failed. Known issue: Silver lab_results "
        "currently contains insurance-provider fields. Correct upstream ingestion; "
        "do not bypass this check. No Gold table was written."
    )


In [0]:
lab = lab_df.select(
    F.col("hospital_id").cast(IntegerType()).alias("hospital_id"),
    F.col("patient_id").cast(IntegerType()).alias("patient_id"),
    F.col("doctor_id").cast(IntegerType()).alias("doctor_id"),
    F.to_date("test_date").alias("test_date"),
    F.trim(F.col("result_status").cast(StringType())).alias("result_status"),
)

if lab.filter(F.col("test_date").isNull()).limit(1).count():
    raise ValueError("Invalid or NULL test_date found. No Gold table was written.")

daily = (
    lab.groupBy("hospital_id", "test_date")
    .agg(
        F.count(F.lit(1)).alias("total_tests"),
        F.countDistinct("patient_id").alias("unique_patients_tested"),
        F.countDistinct("doctor_id").alias("active_doctor_count"),
        F.sum(F.when(F.col("result_status") == "Abnormal", 1).otherwise(0)).alias("abnormal_test_count"),
        F.sum(F.when(F.col("result_status") == "Critical", 1).otherwise(0)).alias("critical_test_count"),
    )
    .withColumnRenamed("test_date", "summary_date")
)


In [0]:
h = hospital_df.select("hospital_id", "hospital_name")
c = calendar_df.select(
    F.col("date_key").cast("string").alias("calendar_date_key"),
    F.to_date("date").alias("calendar_date"),
    "day_name",
    "is_weekend",
)

daily = daily.join(h, "hospital_id", "left")
daily = daily.withColumn("join_date_key", F.date_format("summary_date", "yyyyMMdd"))

daily = (
    daily.join(c, daily.join_date_key == c.calendar_date_key, "left")
    .drop("join_date_key", "calendar_date_key", "calendar_date")
    .withColumn("day_name", F.coalesce("day_name", F.date_format("summary_date", "EEEE")))
    .withColumn("is_weekend", F.coalesce("is_weekend", F.dayofweek("summary_date").isin([1, 7])))
)


In [0]:
daily = (
    daily
    .withColumn("abnormal_rate_pct",
        F.when(F.col("total_tests") > 0, F.col("abnormal_test_count") * 100.0 / F.col("total_tests")))
    .withColumn("critical_rate_pct",
        F.when(F.col("total_tests") > 0, F.col("critical_test_count") * 100.0 / F.col("total_tests")))
)

w = Window.partitionBy("hospital_id").orderBy("summary_date")
daily = daily.withColumn("prev_day_total_tests", F.lag("total_tests").over(w))
daily = daily.withColumn(
    "test_volume_change_pct",
    F.when(
        F.col("prev_day_total_tests").isNull() | (F.col("prev_day_total_tests") == 0),
        F.lit(None).cast("double")
    ).otherwise(
        (F.col("total_tests") - F.col("prev_day_total_tests")) * 100.0 / F.col("prev_day_total_tests")
    )
)


In [0]:
if not ALLOW_CREATE_TARGET:
    print("PREVIEW ONLY: no table was written.")
else:
    if spark.catalog.tableExists(TARGET):
        raise ValueError(f"Target already exists; refusing to overwrite: {TARGET}")
    (gold_df.write.format("delta").mode("errorifexists").saveAsTable(TARGET))
    print("Created new table:", TARGET)

print("Source Silver tables were not modified.")


In [0]:
gold_df = daily.select(
    F.col("hospital_id").cast("int").alias("hospital_id"),
    F.col("hospital_name").cast("string").alias("hospital_name"),
    F.col("summary_date").cast("date").alias("summary_date"),
    F.col("day_name").cast("string").alias("day_name"),
    F.col("is_weekend").cast("boolean").alias("is_weekend"),
    F.col("total_tests").cast("int").alias("total_tests"),
    F.col("unique_patients_tested").cast("int").alias("unique_patients_tested"),
    F.col("active_doctor_count").cast("int").alias("active_doctor_count"),
    F.col("abnormal_test_count").cast("int").alias("abnormal_test_count"),
    F.col("critical_test_count").cast("int").alias("critical_test_count"),
    F.round("abnormal_rate_pct", 2).cast(DecimalType(5, 2)).alias("abnormal_rate_pct"),
    F.round("critical_rate_pct", 2).cast(DecimalType(5, 2)).alias("critical_rate_pct"),
    F.col("prev_day_total_tests").cast("int").alias("prev_day_total_tests"),
    F.round("test_volume_change_pct", 2).cast(DecimalType(6, 2)).alias("test_volume_change_pct"),
)


In [0]:
duplicate_keys = (
    gold_df.groupBy("hospital_id", "summary_date")
    .count().filter(F.col("count") > 1).count()
)
if duplicate_keys:
    raise ValueError(f"Duplicate hospital/date keys found: {duplicate_keys}")

print("Gold preview rows:", gold_df.count())
print("Duplicate grain keys:", duplicate_keys)
display(gold_df.orderBy("hospital_id", "summary_date").limit(50))


In [0]:

import re

# ============================================================
# 1. CATALOG CONFIGURATION
# ============================================================

dbutils.widgets.text("catalog", "databrickscaresync")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: {catalog}")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"

print(f"[SOURCE] {source_schema}")
print(f"[TARGET] {target_table}")


# ============================================================
# 2. VALIDATE SOURCE TABLES
# ============================================================

required_columns = {
    "lab_results": [
        "hospital_id",
        "patient_id",
        "doctor_id",
        "test_date",
        "result_status"
    ],
    "hospitals": [
        "hospital_id",
        "hospital_name"
    ],
    "calendar": [
        "date_key",
        "day_name",
        "is_weekend"
    ]
}

for table_name, expected_columns in required_columns.items():

    full_table = f"{source_schema}.{table_name}"

    if not spark.catalog.tableExists(full_table):
        raise ValueError(f"Table not found: {full_table}")

    actual_columns = {
        field.name.lower()
        for field in spark.table(full_table).schema.fields
    }

    missing_columns = [
        col for col in expected_columns
        if col.lower() not in actual_columns
    ]

    if missing_columns:
        raise ValueError(
            f"Schema mismatch in {full_table}. "
            f"Missing columns: {missing_columns}. "
            f"Available columns: {sorted(actual_columns)}"
        )

print("[VALIDATION] All source tables verified.")


# ============================================================
# 3. CREATE GOLD SCHEMA
# ============================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold"
)


# ============================================================
# 4. DAILY HOSPITAL SUMMARY
# ============================================================

select_sql = f"""

WITH daily AS (

    SELECT
        hospital_id,

        CAST(test_date AS DATE) AS summary_date,

        COUNT(*) AS total_tests,

        COUNT(DISTINCT patient_id)
            AS unique_patients_tested,

        COUNT(DISTINCT doctor_id)
            AS active_doctor_count,

        COUNT_IF(result_status = 'Abnormal')
            AS abnormal_test_count,

        COUNT_IF(result_status = 'Critical')
            AS critical_test_count

    FROM {source_schema}.lab_results

    GROUP BY
        hospital_id,
        CAST(test_date AS DATE)
),

with_lag AS (

    SELECT
        d.*,

        LAG(d.total_tests) OVER (
            PARTITION BY d.hospital_id
            ORDER BY d.summary_date
        ) AS prev_day_total_tests

    FROM daily d
)

SELECT

    CAST(w.hospital_id AS INT)
        AS hospital_id,

    CAST(h.hospital_name AS STRING)
        AS hospital_name,

    w.summary_date
        AS summary_date,

    CAST(c.day_name AS STRING)
        AS day_name,

    CAST(c.is_weekend AS BOOLEAN)
        AS is_weekend,

    CAST(w.total_tests AS INT)
        AS total_tests,

    CAST(w.unique_patients_tested AS INT)
        AS unique_patients_tested,

    CAST(w.active_doctor_count AS INT)
        AS active_doctor_count,

    CAST(w.abnormal_test_count AS INT)
        AS abnormal_test_count,

    CAST(w.critical_test_count AS INT)
        AS critical_test_count,

    CAST(
        w.abnormal_test_count * 100.0 /
        NULLIF(w.total_tests, 0)
        AS DECIMAL(5,2)
    ) AS abnormal_rate_pct,

    CAST(
        w.critical_test_count * 100.0 /
        NULLIF(w.total_tests, 0)
        AS DECIMAL(5,2)
    ) AS critical_rate_pct,

    CAST(w.prev_day_total_tests AS INT)
        AS prev_day_total_tests,

    CASE

        WHEN w.prev_day_total_tests IS NULL
          OR w.prev_day_total_tests = 0

        THEN NULL

        ELSE CAST(
            (w.total_tests - w.prev_day_total_tests)
            * 100.0 / w.prev_day_total_tests
            AS DECIMAL(6,2)
        )

    END AS test_volume_change_pct

FROM with_lag w

LEFT JOIN {source_schema}.hospitals h
    ON w.hospital_id = h.hospital_id

LEFT JOIN {source_schema}.calendar c
    ON c.date_key = CAST(
        date_format(w.summary_date, 'yyyyMMdd')
        AS INT
    )

"""


# ============================================================
# 5. CREATE OR REPLACE GOLD TABLE
# ============================================================

spark.sql(
    f"CREATE OR REPLACE TABLE {target_table} AS {select_sql}"
)

print(f"[DONE] Rebuilt {target_table}")


# ============================================================
# 6. DATA QUALITY CHECKS
# ============================================================

check = spark.sql(f"""

SELECT

    (
        SELECT COUNT(*)
        FROM {source_schema}.lab_results
    ) AS source_tests,

    SUM(total_tests) AS gold_total_tests,

    COUNT(*) AS gold_rows,

    COUNT(DISTINCT hospital_id, summary_date)
        AS gold_distinct_hospital_days

FROM {target_table}

""").first()

print("[DATA QUALITY CHECK]")
print(check)


# ============================================================
# 7. ASSERTIONS
# ============================================================

assert (
    check.gold_rows == check.gold_distinct_hospital_days
), "Duplicate hospital/day rows in Gold"

assert (
    check.gold_total_tests == check.source_tests
), "Source and Gold test counts do not match"

print("[SUCCESS] All data quality checks passed.")


# ============================================================
# 8. DISPLAY FINAL GOLD TABLE
# ============================================================

display(
    spark.table(target_table)
    .orderBy("hospital_id", "summary_date")
    .limit(10)
)


In [0]:

catalog = dbutils.widgets.get("catalog").strip()

print("\n========== SILVER TABLES ==========")

spark.sql(f"SHOW TABLES IN {catalog}.silver").show(100, False)

print("\n========== BRONZE TABLES ==========")

spark.sql(f"SHOW TABLES IN {catalog}.bronze").show(100, False)

print("\n========== BRONZE TABLE SCHEMAS ==========")

bronze_tables = spark.sql(
    f"SHOW TABLES IN {catalog}.bronze"
).collect()

for row in bronze_tables:
    table_name = row.tableName
    full_table = f"{catalog}.bronze.{table_name}"

    print(f"\nTABLE: {full_table}")
    print("COLUMNS:", spark.table(full_table).columns)


In [0]:

catalog = dbutils.widgets.get("catalog").strip()
source_schema = f"{catalog}.silver"

tables = spark.sql(
    f"SHOW TABLES IN {source_schema}"
).collect()

print(f"TOTAL SILVER TABLES: {len(tables)}")

for row in tables:
    table_name = row.tableName
    full_table = f"{source_schema}.{table_name}"

    print(f"\n{'='*70}")
    print(f"TABLE: {full_table}")
    print("="*70)

    columns = spark.table(full_table).columns

    print("COLUMNS:", columns)


In [0]:

import re

# ============================================================
# 1. CATALOG CONFIGURATION
# ============================================================

dbutils.widgets.text("catalog", "databrickscaresync")
catalog = dbutils.widgets.get("catalog").strip()

# Validate catalog identifier
if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: '{catalog}'")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"

print(f"[SOURCE] {source_schema}")
print(f"[TARGET] {target_table}")


# ============================================================
# 2. VERIFY SOURCE TABLES
# ============================================================

required_columns = {
    "lab_results": [
        "hospital_id",
        "patient_id",
        "doctor_id",
        "test_date",
        "result_status"
    ],
    "hospitals": [
        "hospital_id",
        "hospital_name"
    ],
    "calendar": [
        "date_key",
        "day_name",
        "is_weekend"
    ]
}

for table_name, expected_columns in required_columns.items():

    full_table = f"{source_schema}.{table_name}"

    if not spark.catalog.tableExists(full_table):
        raise ValueError(f"Source table does not exist: {full_table}")

    actual_columns = {
        field.name.lower()
        for field in spark.table(full_table).schema.fields
    }

    missing_columns = [
        column for column in expected_columns
        if column.lower() not in actual_columns
    ]

    if missing_columns:
        raise ValueError(
            f"Schema mismatch in {full_table}. "
            f"Missing columns: {missing_columns}. "
            f"Available columns: {sorted(actual_columns)}"
        )

print("[VALIDATION] All source tables passed schema validation.")


# ============================================================
# 3. CREATE GOLD SCHEMA
# ============================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")


# ============================================================
# 4. DAILY AGGREGATION + WINDOW FUNCTION + JOINS
# ============================================================

select_sql = f"""
WITH daily AS (

    -- One record per hospital per day

    SELECT
        hospital_id,
        CAST(test_date AS DATE) AS summary_date,

        COUNT(*) AS total_tests,

        COUNT(DISTINCT patient_id)
            AS unique_patients_tested,

        COUNT(DISTINCT doctor_id)
            AS active_doctor_count,

        COUNT_IF(result_status = 'Abnormal')
            AS abnormal_test_count,

        COUNT_IF(result_status = 'Critical')
            AS critical_test_count

    FROM {source_schema}.lab_results

    GROUP BY
        hospital_id,
        CAST(test_date AS DATE)
),

with_lag AS (

    -- Previous available day's test volume per hospital

    SELECT
        d.*,

        LAG(d.total_tests) OVER (
            PARTITION BY d.hospital_id
            ORDER BY d.summary_date
        ) AS prev_day_total_tests

    FROM daily d
)

SELECT

    CAST(w.hospital_id AS INT)
        AS hospital_id,

    CAST(h.hospital_name AS STRING)
        AS hospital_name,

    w.summary_date
        AS summary_date,

    CAST(c.day_name AS STRING)
        AS day_name,

    CAST(c.is_weekend AS BOOLEAN)
        AS is_weekend,

    CAST(w.total_tests AS INT)
        AS total_tests,

    CAST(w.unique_patients_tested AS INT)
        AS unique_patients_tested,

    CAST(w.active_doctor_count AS INT)
        AS active_doctor_count,

    CAST(w.abnormal_test_count AS INT)
        AS abnormal_test_count,

    CAST(w.critical_test_count AS INT)
        AS critical_test_count,

    CAST(
        w.abnormal_test_count * 100.0 / NULLIF(w.total_tests, 0)
        AS DECIMAL(5,2)
    ) AS abnormal_rate_pct,

    CAST(
        w.critical_test_count * 100.0 / NULLIF(w.total_tests, 0)
        AS DECIMAL(5,2)
    ) AS critical_rate_pct,

    CAST(w.prev_day_total_tests AS INT)
        AS prev_day_total_tests,

    CASE
        WHEN w.prev_day_total_tests IS NULL
          OR w.prev_day_total_tests = 0
        THEN NULL

        ELSE CAST(
            (w.total_tests - w.prev_day_total_tests)
            * 100.0 / w.prev_day_total_tests
            AS DECIMAL(6,2)
        )
    END AS test_volume_change_pct

FROM with_lag w

LEFT JOIN {source_schema}.hospitals h
    ON w.hospital_id = h.hospital_id

LEFT JOIN {source_schema}.calendar c
    ON c.date_key = CAST(
        date_format(w.summary_date, 'yyyyMMdd')
        AS INT
    )
"""


# ============================================================
# 5. CREATE OR REPLACE GOLD TABLE
# ============================================================

spark.sql(
    f"CREATE OR REPLACE TABLE {target_table} AS {select_sql}"
)

print(f"[DONE] Rebuilt {target_table}")


# ============================================================
# 6. DATA QUALITY VALIDATION
# ============================================================

check = spark.sql(f"""
SELECT

    (SELECT COUNT(*)
     FROM {source_schema}.lab_results) AS source_tests,

    SUM(total_tests) AS gold_total_tests,

    COUNT(*) AS gold_rows,

    COUNT(DISTINCT hospital_id, summary_date)
        AS gold_distinct_hospital_days

FROM {target_table}
""").first()

print("[DATA QUALITY CHECK]")
print(check)


# ============================================================
# 7. ASSERTIONS
# ============================================================

assert (
    check.gold_rows == check.gold_distinct_hospital_days
), "Duplicate hospital/day rows in Gold table"

assert (
    check.gold_total_tests == check.source_tests
), "Sum of total_tests differs from Silver lab_results row count"

print("[SUCCESS] All data quality checks passed.")


# ============================================================
# 8. DISPLAY FINAL GOLD DATA
# ============================================================

display(
    spark.table(target_table)
    .orderBy("hospital_id", "summary_date")
    .limit(10)
)


In [0]:
import re

dbutils.widgets.text("catalog", "databrickscaresync")
catalog = dbutils.widgets.get("catalog").strip()

# Guard against injection through the catalog widget
if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: '{catalog}'")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"
print(f"[SOURCE] {source_schema}  ->  [TARGET] {target_table}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

# Inspect all three Silver tables
tables = ["lab_results", "hospitals", "calendar"]

for table in tables:
    full_table = f"{source_schema}.{table}"

    print(f"\n{'='*60}")
    print(f"TABLE: {full_table}")
    print(f"{'='*60}")

    if spark.catalog.tableExists(full_table):
        spark.sql(f"DESCRIBE TABLE {full_table}").show(truncate=False)
        print("Sample records:")
        display(spark.table(full_table).limit(3))
    else:
        print(f"TABLE NOT FOUND: {full_table}")

In [0]:
print("LAB RESULTS SCHEMA")

spark.sql(
    f"DESCRIBE TABLE {source_schema}.lab_results"
).show(100, truncate=False)

print("LAB RESULTS SAMPLE")

display(
    spark.table(f"{source_schema}.lab_results").limit(5)
)

In [0]:
check = spark.sql(f"""
SELECT
  (SELECT COUNT(*) FROM {source_schema}.lab_results)       AS source_tests,
  SUM(total_tests)                                         AS gold_total_tests,
  COUNT(*)                                                 AS gold_rows,
  COUNT(DISTINCT hospital_id, summary_date)                AS gold_distinct_hospital_days
FROM {target_table}
""").first()
print(check)

# Grain: one row per hospital per day; every lab test counted exactly once
assert check.gold_rows == check.gold_distinct_hospital_days, "Duplicate hospital/day rows in gold"
assert check.gold_total_tests == check.source_tests, "Sum of total_tests differs from silver.lab_results row count"

display(spark.table(target_table).orderBy("hospital_id", "summary_date").limit(10))